## Проверка модели

In [3]:
import torch
from ml.config import env
from ml.logger_config import log_event
from ml.models import model_detector

model_detector.to(env.device)
model_detector.eval()

test_tsr = torch.randn(1, 1, 640, 640).to(env.device)
x1 = model_detector(test_tsr)

model_detector.train()
x2 = model_detector(test_tsr)

if isinstance(x2, dict):
    print('# train out')
    print('boxes', x2['boxes'].shape)
    print('scores', x2['scores'].shape)
    print('feats', x2['feats'][0].shape, end='\n\n')

if isinstance(x1, tuple):
    print('# eval out')
    print(f"output[0]: {x1[0].shape}")
    print(f"output[1]: {x1[1].keys()}")
    print(f"output[1]: {x1[1]['boxes'].shape}")


# train out
boxes torch.Size([1, 64, 8400])
scores torch.Size([1, 1, 8400])
feats torch.Size([1, 64, 80, 80])

# eval out
output[0]: torch.Size([1, 5, 8400])
output[1]: dict_keys(['boxes', 'scores', 'feats'])
output[1]: torch.Size([1, 64, 8400])


## Гиперпарамтеры

In [7]:
from ml.config import WORKDIR, env
from ml.models import model_detector, model_detector_code

from ultralytics.utils.loss import v8DetectionLoss
from types import SimpleNamespace
from torch.optim import AdamW
from torch.optim.lr_scheduler import MultiStepLR
from datetime import datetime


model_detector.to(env.device)
loss_func = v8DetectionLoss(model_detector)
hyp = SimpleNamespace(box=7.5, cls=0.5, dfl=1.5)
loss_func.hyp = hyp

opt = AdamW(model_detector.parameters(), lr=0.01, weight_decay=5e-4)
lr_sched = MultiStepLR(opt, milestones=[5, 35, 50], gamma=0.1)

epochs = 60
batch_size_train = 32
batch_size_val = 16
batch_size_test = 32
dataload_workers = 12

threshold_loss = 0.03
early_stopping = 12
models_dir = WORKDIR / 'ml' / 'model_weights' / 'detector' / f'{datetime.now().strftime("%Y%m%d-%H%M%S")}'
models_dir.mkdir(parents=True)


#### Функции для графиков по лоссам и метрикам

In [8]:
from ml.utils import plot_curves


def plot_validation_metrics(history_arg, save_path=None):
    values_list = history_arg["general_metrics"]
    curves = [
        {
            "label": "Val Loss",
            "values": values_list["val_loss_list"],
            "color": "red"
        },
        {
            "label": "mAP@0.5",
            "values": values_list["map50_list"],
            "color": "blue"
        },
        {
            "label": "mAP@0.5:0.95",
            "values": values_list["map5095_list"],
            "color": "green"
        }
    ]
    print(curves)

    plot_curves(
        curves=curves,
        title="Validation Loss & Metrics",
        ylabel="Loss / mAP",
        save_path=save_path
    )

def plot_training_dynamics(history_arg, save_path=None):
    values_list = history_arg["general_metrics"]
    curves = [
        {
            "label": "Train Loss",
            "values": values_list["train_loss_list"],
            "color": "orange"
        },
        {
            "label": "Val Loss",
            "values": values_list["val_loss_list"],
            "color": "red"
        },
        {
            "label": "Learning Rate",
            "values": history_arg["lr"],
            "color": "purple"
        }
    ]
    print(curves)

    plot_curves(
        curves=curves,
        title="Training Dynamics",
        ylabel="Loss / LR",
        save_path=save_path
    )


#### Датасет, Даталоадеры

In [9]:
from ml.dataclass_detector import OCRDetectorDataset
from torch.utils.data import DataLoader
from ml.config import WORKDIR


train_dset = OCRDetectorDataset(WORKDIR / 'dataset' / 'iam-form-stratified' / 'train', 'train')
val_dset = OCRDetectorDataset(WORKDIR / 'dataset' / 'iam-form-stratified' / 'val', 'val')
test_dset = OCRDetectorDataset(WORKDIR / 'dataset' / 'iam-form-stratified' / 'test', 'val')

train_loader = DataLoader(
    dataset=train_dset,
    batch_size=batch_size_train,
    shuffle=True,
    num_workers=dataload_workers,
    collate_fn=OCRDetectorDataset.collate_fn,
    pin_memory=True,
    persistent_workers=True
)
val_loader = DataLoader(
    dataset=val_dset,
    batch_size=batch_size_val,
    shuffle=True,
    num_workers=dataload_workers,
    collate_fn=OCRDetectorDataset.collate_fn,
    pin_memory=True,
    persistent_workers=True
)
test_loader = DataLoader(
    dataset=val_dset,
    batch_size=batch_size_test,
    shuffle=True,
    num_workers=dataload_workers,
    collate_fn=OCRDetectorDataset.collate_fn,
    pin_memory=True,
    persistent_workers=True
)

WARNING  | D26-01-2026 T23:51:19 | ml\dataclass_detector.py: def create_data(): line - 164 В наборе данных 960 размеченных изображений
WARNING  | D26-01-2026 T23:51:19 | ml\dataclass_detector.py: def create_data(): line - 164 В наборе данных 240 размеченных изображений
WARNING  | D26-01-2026 T23:51:19 | ml\dataclass_detector.py: def create_data(): line - 164 В наборе данных 326 размеченных изображений


## Обучение & Валидация

In [10]:
import torch
from ultralytics.utils.nms import non_max_suppression
from ultralytics.utils.metrics import ap_per_class, box_iou
from ml.config import env
from tqdm import tqdm
from ml.models import model_detector



log_event(f'\033[34mОбучение началось\033[0m | Эпохи: \033[33m{epochs}\033[0m')

train_loss, val_loss, lr_list, map50_list, map5095_list, min_loss = [], [], [], [], [], None
plateau_loss_epochs = 0
iouv = torch.linspace(0.5, 0.95, 10).to(env.device)

for epoch in range(1, epochs + 1):

    model_detector.train()

    last_losses_train = []
    train_loop = tqdm(train_loader, leave=False, desc=f'Training \033[34m#{epoch}\033[0m')
    for img, targets, words in train_loop:
        "Предсказание модели на батч"
        img = img.to(env.device, non_blocking=True)
        targets = targets.to(env.device, non_blocking=True)
        batch_dict = {
            "batch_idx": targets[:, 0],
            "cls": targets[:, 1].long(),
            "bboxes": targets[:, 2:]
        }

        preds = model_detector(img)
        total_loss_tsr, losses_tsr = loss_func(preds, batch_dict)
        backward_tsr = total_loss_tsr.mean()

        opt.zero_grad()
        backward_tsr.backward()
        opt.step()

        "Считаем лосс"
        total_loss = backward_tsr.detach().item()
        losses = [loss_tsr.item() for loss_tsr in losses_tsr]
        losses.append(total_loss)
        last_losses_train = losses

        train_loss.append(losses)

    log_event(f"\033[32mTRAINING\033[0m | Epoch {epoch}, train_loss={last_losses_train[-1]:.4f}, box_loss={last_losses_train[0]:.4f}, cls_loss={last_losses_train[1]:.4f}, dfl_loss={last_losses_train[2]:.4f}")
    train_loss.append(last_losses_train)


    model_detector.eval()
    with torch.no_grad():
        last_losses_val = []
        stats = []

        val_loop = tqdm(val_loader, leave=False, desc=f'Validation \033[36m#{epoch}\033[0m')
        for img, targets, words in val_loader:

            img = img.to(env.device, non_blocking=True)
            targets = targets.to(env.device, non_blocking=True)

            batch_dict = {
                "batch_idx": targets[:, 0],
                "cls": targets[:, 1].long(),
                "bboxes": targets[:, 2:]
            }

            preds = model_detector(img)
            total_loss_tsr, losses_tsr = loss_func(preds, batch_dict)

            losses = [loss_tsr.item() for loss_tsr in losses_tsr]
            total_loss = total_loss_tsr.mean().detach().item()
            losses.append(total_loss)

            last_losses_val = losses

            preds_nms = non_max_suppression(
                preds,
                conf_thres=0.25,
                iou_thres=0.45,
                agnostic=True,
                max_det=600,
                nc=model_detector.nc
            )

            "Метрики"
            for i, pred in enumerate(preds_nms):
                # Фильтруем таргеты: берем только те, где batch_idx == i
                gt_mask = (targets[:, 0] == i)
                gt = targets[gt_mask]  # Теперь здесь [num_gt, 6] (batch_idx, cls, x, y, w, h)

                nl = gt.shape[0]
                npr = pred.shape[0]
                correct = torch.zeros(npr, len(iouv), dtype=torch.bool, device=env.device)

                # Подготовка правильных координат GT для сопоставления

                # В targets xywh (нормализованные), а NMS выдает xyxy (в пикселях)
                if nl:
                    # 1. Переводим GT из xywh в xyxy
                    # 2. Денормализуем (умножаем на размер изображения, например 640)
                    h, w = 640, 640
                    gt_boxes = gt[:, 2:].clone()

                    # Конвертация xywh -> xyxy
                    gn = torch.tensor([w, h, w, h], device=env.device)
                    gt_boxes[:, [0, 2]] *= w # x, w
                    gt_boxes[:, [1, 3]] *= h # y, h

                    x, y, w_gt, h_gt = gt_boxes.unbind(1)
                    gt_xyxy = torch.stack([
                        x - w_gt / 2, y - h_gt / 2,
                        x + w_gt / 2, y + h_gt / 2
                    ], dim=1)
                else:
                    gt_xyxy = torch.zeros((0, 4), device=env.device)

                if nl:
                    # Теперь gt_xyxy имеет размерность [nl, 4]
                    iou = box_iou(pred[:, :4], gt_xyxy)
                    for j in range(len(iouv)):
                        correct[:, j] = (iou >= iouv[j]).any(1)
                if npr > 0:
                    # stats ожидает:
                    # 1. correct [npr, 10]
                    # 2. conf [npr]
                    # 3. pred_cls [npr]
                    # 4. target_cls [nl] <-- ВАЖНО: только истинные классы изображения
                    stats.append((
                        correct.cpu(),
                        pred[:, 4].cpu(),
                        pred[:, 5].cpu(),
                        gt[:, 1].cpu()
                    ))

    lr_sched.step()
    lr = lr_sched.get_last_lr()
    lr_list.append(lr)

    "Подсчёт метрик за эпоху"
    if len(stats):
        tp, conf, pred_cls, target_cls = zip(*stats)

        # Конкатенируем (tp — это список тензоров [npr, 10], остальные [npr] или [nl])
        stats_cat = [
            torch.cat(tp, 0).numpy(),
            torch.cat(conf, 0).numpy(),
            torch.cat(pred_cls, 0).numpy(),
            torch.cat(target_cls, 0).numpy()
        ]

        if stats_cat[0].any() or len(stats_cat[3]) > 0:
            results = ap_per_class(*stats_cat, names=model_detector.names)
            tp, fp, p, r, f1, ap, *_ = results

            map50 = ap[:, 0].mean()
            map5095 = ap.mean()
        else:
            map50, map5095 = 0.0, 0.0

    else:
        map50, map5095 = 0.0, 0.0

    map50_list.append(map50)
    map5095_list.append(map5095)

    log_event(f"\033[34mVALIDATION\033[0m Epoch {epoch} | val_loss={last_losses_val[-1]:.4f}, box_loss={last_losses_val[0]:.4f}, cls_loss={last_losses_val[1]:.4f}, dfl_loss={last_losses_val[2]:.4f} | mAP@0.5={map50:.4f} | mAP@0.5:0.95={map5095:.4f}")

    history = {
        "general_metrics": {
            'val_loss_list': val_loss,
            'train_loss_list': train_loss,
            'map50_list': map50_list,
            'map5095_list': map5095_list,
        },
        "train_loss_last": last_losses_train,
        "val_loss_last": val_loss,
        "map50_cur": map50,
        "map5095_cur": map5095,
        "lr": lr_list
    }

    "Сохраняем модель, как только виднеется прогресс"
    min_loss = last_losses_val[-1] if min_loss is None else min_loss
    if min_loss - min_loss * threshold_loss > last_losses_val[-1]:
        min_loss = last_losses_val[-1]

        checkpoint = {
            'model_detector': model_detector_code,
            'state_model': model_detector.state_dict(),
            'state_opt': opt.state_dict(),
            'state_lr_scheduler': lr_sched.state_dict(),
            'save_epoch': epoch,
            'history': history,
        }
        torch.save(checkpoint, models_dir.joinpath(f'model_detector{epoch}.pth'))
        plateau_loss_epochs = 0
        log_event(f'На эпохе - \033[35m{epoch}\033[0m модель сохранена | Ранняя остановка обучения изменена', level='WARNING')

    "Early Stopping"
    if plateau_loss_epochs >= early_stopping:
        log_event(f'\033[31m{'!!!' * 10} Принудительная остановка обучения, нет прогресса {'!!!' * 10}\033[0m', level='WARNING')
        raise Exception("Early Stopping")

    plateau_loss_epochs += 1

plot_training_dynamics(history, models_dir / 'training.png')
plot_validation_metrics(history, models_dir / 'validation.png')

log_event(f'{'>>>' * 10} Обучение завершено {'<<<' * 10}')

Обучение началось | Эпохи: 60


TRAINING | Epoch 1, train_loss=360.4460, box_loss=3.7853, cls_loss=27.7147, dfl_loss=2.2918


Validation #1:   0%|          | 0/15 [00:00<?, ?it/s]

VALIDATION Epoch 1 | val__loss=1829.4475, box_loss=6.9476, cls_loss=331.5531, dfl_loss=4.5207 | mAP@0.5=0.0000 | mAP@0.5:0.95=0.0000



Training #2: 100%|██████████| 30/30 [00:32<00:00,  2.39it/s]
                                                            

...

VALIDATION Epoch 54 | val__loss=21.7761, box_loss=1.0769, cls_loss=2.0467, dfl_loss=0.9594 | mAP@0.5=0.8596 | mAP@0.5:0.95=0.6052


TRAINING | Epoch 55, train_loss=44.1460, box_loss=1.0833, cls_loss=2.1104, dfl_loss=0.9450


Validation #55:   0%|          | 0/15 [00:00<?, ?it/s]
                                                      

VALIDATION Epoch 55 | val__loss=22.3665, box_loss=1.0756, cls_loss=2.1455, dfl_loss=0.9726 | mAP@0.5=0.8631 | mAP@0.5:0.95=0.6081
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!! Принудительная остановка обучения, нет прогресса !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


Exception: Early Stopping